# Dedicated [O I] Coefficient Transfer

**Status.** Experiment on branch `oi-dedicated-mlp-v1`. Nothing here changes the deployed
SkyML pipeline.

## Problem statement

In SkyML the sky [O I] 6300/6364 doublet subtracted from a science exposure is purely the
deployed ensemble's prediction of one coefficient, `ATOM_Or`. Nothing measured on the
exposure corrects it: `sky_line_scaling` keeps `ATOM_Or` in `FIXED_FAMILIES`, and the
sky-arm residual transfer zeroes the line cores, because science [O I] sits at the sky
wavelength. Any error in the transferred amplitude therefore reaches the subtracted
spectrum unchanged.

In the deployed dual-encoder group-head network, `ATOM_Or` has no weights of its own. It is
one of four outputs of the `ionospheric` head (with N I 5199 and O I 7774/8446), on top of
an encoder and trunk shared by all 388 coefficients, ~357 of them OH.

This notebook asks one question: **does a small network dedicated to the O I lines transfer
their science-pointing amplitude better than the deployed ensemble?** The O I lines are
grouped by emission physics:

| Group | Coefficients | Mechanism | Height |
|---|---|---|---|
| `or` | `ATOM_Or` ([O I] 6300/6364) | O2+ dissociative recombination, O(1D) | ~250 km |
| `orc` | `ATOM_Orc_OI0777`, `ATOM_Orc_OI0845` | O+ radiative recombination | ~300-350 km |
| `og` | `ATOM_Og` ([O I] 5577) | mainly Barth mechanism, O(1S) | ~97 km |
| `all` | all four in one network | | |

**Scientific contract.** The experiment works in coefficient space on the existing
decomposition products. No spectrum is resampled, smoothed, cropped or masked. The pixel
check in the last section uses the native 12,401-pixel grid and replaces only the
`ATOM_Or` amplitude.

## Method

*Corpus.* `lvmsframe_median_stack_1.3.2_knox_gaia1over100`, decomposition
`palacecorr_aijc_vnf_split_zodi_lsf_spline2d_solarblue`, 18,664 exposures. The triplet,
filters and context augments are those of the deployed training notebook, including the
moon/zodi reversal gate on the full `*_decomp_*` files (it removes no rows here).
13,682 rows remain.

*Label cleaning.* Science [O I] can include nebular emission from the target, because
6300/6364 is not in the decomposition's science-line mask. Rows whose science `ATOM_Or`
differs from the geometric mean of the two sky arms by more than 0.3 dex (a factor of 2),
and the non-science tile 11111, are removed from training, validation and the primary test
score. Scores on the full test split are reported as well.

*Split.* The deployed notebook's night-level, moon-phase-stratified split with seed 42 on
this filtered set: 10,833 / 1,412 / 1,437 rows (train / validation / test). Section 3.1
shows that it agrees with the split the deployed ensemble was trained on.

*Networks.* Each variant is a two-layer MLP (128, 64; LayerNorm, GELU, dropout 0.1), an
ensemble of 5 seeds. Inputs: log10 of the four O I coefficients in both sky arms and the
deployed ensemble's 42 context features for the science, near and far pointings. Target:
log10(science) - log10(near) for the group's coefficients, weighted by the science
coefficient error in log space with a 0.02 dex floor. AdamW, lr 1e-3, weight decay 1e-4,
batch 256, early stopping on validation loss (patience 40, at most 400 epochs).

*Baselines.* Near-arm copy, geometric mean of the two sky arms, and the deployed ensemble
`mlp_ensemble_1.3.2_gaia_stars_solarblue_knox.pt` evaluated on the same rows.

*Commands* (from `skysub/`, `LVMCORE_DIR` and `SAS_BASE_DIR` set):

```bash
python -m experiments_oi_dedicated_mlp_v1.prepare --corpus-dir <corpus dir> \
    --ensemble ../mlp_ensemble_1.3.2_gaia_stars_solarblue_knox.pt \
    --out experiments_oi_dedicated_mlp_v1/outputs/oi_table.npz
python -m experiments_oi_dedicated_mlp_v1.train \
    --table experiments_oi_dedicated_mlp_v1/outputs/oi_table.npz \
    --out-dir experiments_oi_dedicated_mlp_v1/outputs/run_v1
# section 4, from the lvmsky repo root
python -m skysub.experiments_oi_dedicated_mlp_v1.exposure <lvmCFrame-00014771.fits> \
    --ensemble mlp_ensemble_1.3.2_gaia_stars_solarblue_knox.pt \
    --models skysub/experiments_oi_dedicated_mlp_v1/outputs/run_v1/oi_models.pt \
    --out-dir skysub/experiments_oi_dedicated_mlp_v1/outputs/exposure_14771 --jobs 6
```


## Changelog and measured record

**2026-11-20, run_v1.** Held-out test, cleaned rows (n = 1,361, 74 nights). Median
|log10 error| (MAD), relative change against the deployed ensemble and its 95% interval
from a 2,000-draw bootstrap over test nights:

| Line | Deployed ensemble | Dedicated MLP | Change | 95% interval |
|---|---:|---:|---:|---:|
| [O I] 6300/6364 (`or`) | 0.0104 | 0.0080 | -22.7% | [-28.4%, -13.6%] |
| O I 7774 (`all`) | 0.0203 | 0.0164 | -19.4% | [-27.4%, -8.8%] |
| O I 8446 (`all`) | 0.0205 | 0.0161 | -21.6% | [-28.7%, -13.1%] |
| [O I] 5577 (`og`) | 0.0049 | 0.0038 | -24.0% | [-28.7%, -17.3%] |

For [O I] 6300/6364 the amplitude-weighted relative RMS falls from 4.78% to 4.09%, and the
dedicated network is closer to the truth on 61% of test rows. The single `all` network is
worse than the dedicated `or` and `og` networks on those lines, and slightly better than
`orc` on the two recombination lines.

[O I] 6300 left after subtraction (section 3, 133 every10 test exposures, +/-3 A of the line,
only the [O I] height changed in the science sky model): median |leftover| 2.60% of the line
flux for the deployed ensemble, 1.80% for the `or` network, against 1.06% when the height is
fitted on the science spectrum itself; 90th percentile 8.3% vs 7.0%.

Split check: on [O I] 5577, 89% of train nights but only 6% of test nights give the deployed
ensemble a train-like per-night error (MAD < 0.0025 dex), so the recomputed split matches the
one it was trained on and the comparison is on nights it did not see.

Exposure 14771 through SkyML (section 4), [O I] 6300 amplitude: science median fit 178.8,
deployed 186.2 (+4.1%), O I network 179.5 (+0.4%). Median line flux left in the Science fibers,
as a fraction of the subtracted sky line (+/-3 A core, 6-12 A sidebands as baseline):

| Line | Deployed ensemble | O I networks |
|---|---:|---:|
| [O I] 5577 | -1.8% | +0.4% |
| [O I] 6300 | -2.9% | +0.2% |
| [O I] 6364 | -3.1% | -0.6% |
| O I 7774 | -3.7% | -1.6% |
| O I 8446 | -3.6% | -1.6% |

Negative means over-subtracted. One exposure from a training night; not a statistical result.

*Reversal gate.* With the full `*_decomp_*` files downloaded, `prepare.py` applies the
deployed notebook's moon/zodi reversal gate. It flags 0 of 18,639 rows, so the table is
identical to the earlier run without the gate, and the moon-model cache is built by the
unchanged shared code with zodi correction `lvm-ecl-2026-09` read from the decomposition
headers. No shared code is modified by this experiment.


In [ ]:
# --- Imports and paths ---
import json
import sys
from pathlib import Path

import numpy as np
from IPython.display import display

SKYSUB = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'mlp_predictor').is_dir())
sys.path.insert(0, str(SKYSUB))
from experiments_oi_dedicated_mlp_v1 import plots
from experiments_oi_dedicated_mlp_v1.train import clean_mask

EXPERIMENT = SKYSUB / 'experiments_oi_dedicated_mlp_v1'
RUN = EXPERIMENT / 'outputs' / 'run_v1'
CORPUS = Path('/Users/ik52/obs/sas/sdsswork/users/u6058164/medians_1.3.2_knox/sky_decomposition_1.3.2_knox')
STEM = 'lvmsframe_median_stack_1.3.2_knox_gaia1over100'
EVERY10 = CORPUS / f'{STEM}_every10.fits'
EVERY10_SCI = CORPUS / f'{STEM}_every10_decomp_sci_palacecorr_aijc_vnf_split_zodi_lsf_spline2d_solarblue.fits'

plots.journal_style()


In [ ]:
# --- Load the prepared table and the run_v1 results ---
t = dict(np.load(EXPERIMENT / 'outputs' / 'oi_table.npz'))
summary = json.loads((RUN / 'results.json').read_text())
preds = dict(np.load(RUN / 'predictions.npz'))
clean = clean_mask(t, summary['outlier_dex'])
test_clean = np.intersect1d(t['test_idx'], np.flatnonzero(clean))
print(summary['rows'])


## 1. Corpus and label cleaning


In [ ]:
# Targets: science ATOM_Or against the two sky arms; the 0.3 dex cut that defines clean rows.
# Look for: a narrow core around zero (sky [O I] transfers well) and tails on both sides.
#           The positive tail is science [O I] on the sky line; the negative tail is a
#           science fit that took less [O I] than the sky arms (or a sky arm with nebular [O I]).
plots.outlier_cut(t, clean, summary['outlier_dex'])


In [ ]:
# Targets: rank correlation of the four O I coefficients in the near sky arm.
# Look for: 6300 vs 7774/8446 well above zero (both trace the F region) but well below one,
#           since the 6300/7774 ratio tracks the F-layer height.
plots.oi_correlations(t, arm='near')


## 2. Results in coefficient space


In [ ]:
# Targets: per line, every method on the cleaned held-out test rows (log10 errors in dex).
# Look for: the dedicated network below the deployed ensemble in MAD, p90 and the
#           amplitude-weighted relative RMS (wRelRMS, which weights bright [O I] rows).
plots.results_table(summary, 'test_clean')


In [ ]:
# Targets: the same table on all test rows, outliers included.
# Look for: the ranking should not change; RMS grows on every method because the outlier
#           rows are labelled with science emission no sky transfer can predict.
plots.results_table(summary, 'test_all')


In [ ]:
# Targets: predicted vs decomposed science ATOM_Or, deployed ensemble vs dedicated network.
# Look for: a tighter cloud around the 1:1 line on the right panel, at both ends of the range.
plots.pred_vs_true(t, preds, test_clean, 'ATOM_Or', ('deployed_ensemble', 'oi_mlp_or'))


In [ ]:
# Targets: median |log10 error| of ATOM_Or against the drivers of the 6300 airglow:
#          Sun altitude (time from twilight), the 285 km van Rhijn factor, F10.7 and Kp.
# Look for: a regime where one method departs from the others; a gain concentrated in
#           one regime points at what the dedicated network learned.
plots.error_vs_context(t, preds, test_clean, 'ATOM_Or')


### 2.1 Does the deployed ensemble see these test nights?

The deployed archive does not store its split, so the split above was recomputed. If it
differed, the deployed ensemble would be scored partly on nights it trained on, which
would favour it. Its own error separates the two cases: on its training nights it is several
times more accurate than on held-out nights.


In [ ]:
# Targets: per-night deployed-ensemble error on [O I] 5577, where its in-training error is most distinct.
# Look for: train nights small, val/test nights large. A large share of test nights with a
#           train-like error would mean the deployed ensemble saw them.
plots.split_overlap_check(t, preds, 'ATOM_Og')


## 3. What is left of [O I] 6300 after subtraction

The coefficient tables above score the predicted line height. Here the same predictions are
put into the sky model of real spectra. Spectra are local only for every tenth exposure
(the every10 file), so this check uses the 133 exposures of the cleaned test nights that are
in every10.

For each exposure, everything in the science sky model except [O I] is kept as fitted; only
the height of the [O I] line changes:

- **Fitted on the science spectrum (best possible).** The [O I] height fitted on the science
  spectrum itself. The pipeline cannot use this, because target [O I] adds to it; it marks
  how well subtraction can work at all, given the line shape.
- **Deployed ensemble** and **[O I] MLP**. The [O I] height predicted from the sky arms.


In [ ]:
# Targets: three test exposures at the 50th, 90th and 99th percentile of [O I] brightness.
#          Black: the observed [O I] line (observed spectrum minus the sky model without [O I]).
#          Coloured: the [O I] line each method would subtract.
# Look for: a coloured line matching the black one; too high = over-subtraction, too low = [O I] left behind.
window = plots.oi_window(EVERY10, EVERY10_SCI, t, test_clean)
plots.oi_example_spectra(window, t, preds)


In [ ]:
# Targets: [O I] 6300 flux left after subtraction within +/-3 A, in % of the line flux, per exposure.
#          Left: distribution (above zero = sky line under-subtracted, below = over-subtracted).
#          Right: typical size, median |leftover|.
# Look for: the [O I] MLP closer to the best-possible bar than the deployed ensemble.
fig, leftover = plots.oi_leftover(window, t, preds)
fig.show()
display(leftover)


## 4. One exposure through SkyML: 14771

The same lvmCFrame (`1.3.2_fix`, tile 1044896, MJD 60392) is run through SkyML twice by
`exposure.py`: once unchanged, and once with the four O I values of the ML coefficient vector
replaced by the dedicated networks, one per emission mechanism (`or` for `ATOM_Or`, `orc` for
`ATOM_Orc_OI0777`/`ATOM_Orc_OI0845`, `og` for `ATOM_Og`). Everything else, including the median
fits, the per-fiber R/Z LSF fits and the residual transfer, runs as deployed. The two products
are `outputs/exposure_14771/{deployed,oi_mlp}/lvmSFrame-skyml-00014771.fits`; the second carries
`OIMLP` and `OIMAP` in its primary header.

This exposure is not in the training table (it is in the corpus but fails the row filters),
but the other 10 exposures of its night are training rows for both the deployed ensemble and
the O I networks, so it is not a held-out-night test.

The table compares the O I amplitudes. The science median fit is the decomposition of this
exposure's own Gaia-clean science median, the quantity both predictors are trained to reproduce.


In [ ]:
# --- O I amplitudes of exposure 14771 (internal 1e14 units) ---
EXPOSURE = EXPERIMENT / 'outputs' / 'exposure_14771'
exposure_paths = {'deployed_ensemble': EXPOSURE / 'deployed' / 'lvmSFrame-skyml-00014771.fits',
                  'oi_mlp_groups': EXPOSURE / 'oi_mlp' / 'lvmSFrame-skyml-00014771.fits'}
display(plots.exposure_coefficients(exposure_paths, list(t['oi_names'])).round(3))


Top row: median over the Science fibers of the delivered `FLUX` (observed minus subtracted sky)
around each O I line. A flat line at the continuum level means the sky line is removed exactly;
a dip means too much sky was subtracted, a bump too little. Gray dashed lines are the median's
error, 1.2533 times the typical per-fiber error over the square root of the fiber count, drawn
around the deployed curve's window median. Bottom row: median over the Science fibers of
SKY(new) - SKY(deployed), the change in what is subtracted.


In [ ]:
plots.exposure_oi_spectra(exposure_paths).show()


## Interpretation boundary

The gain is measured against the science decomposition, whose [O I] amplitude is the
training target; it is not yet measured on sky-subtracted CFrames. Label cleaning removes
rows where the science and sky [O I] differ by more than a factor of 2, which also removes
any genuine sky gradient that large. The
dedicated networks are not wired into SkyML; section 4 replaces the O I values for one exposure
only, from the experiment script.
